# Latency optimization for models and agents — companion notebook

Companion to the deep-dive deck **Microsoft Foundry Latency Optimization for Models and Agents.pptx**, edition
2026-11 (18 slides, in `decks/`). Steps 1–5 look at a single model call; steps 6–10 run one business scenario
through five agent patterns; steps 11–13 cover the agent runtime, retrieval and a latency budget.

**Scenario.** An employee travel desk with three specialists — travel policy, budget and itinerary — answers one
question: *is this trip within policy and budget, and which flights should I book?* The three tools are local
Python functions that return fixed, made-up data at once, so the differences between patterns come from model
calls and orchestration, not from tool latency.

**Reading the numbers.** Every number printed here is a demonstration, not a controlled benchmark: a few runs per
variant, from one client, at one time of day, against one deployment. Client-side times include network and software development kit
(SDK) time; Azure Monitor metrics are measured at the service. Use the steps to show *where* time goes, then measure the
real workload with tracing and percentiles.

**Before you start (Azure side)**
- A Foundry project (not hub-based); its endpoint in `FOUNDRY_PROJECT_ENDPOINT`.
- A chat model deployment that accepts reasoning effort, for example a GPT-5 family model, on **Global Standard**,
  in `FOUNDRY_MODEL_NAME`. For step 5, priority processing needs model version 2025-12-01 or later on a model listed
  for priority processing, and flex processing (preview) needs a model listed for flex processing; other models
  return an error that the step records.
- Your account has the **Foundry User** role on the project.
- *Optional, step 3:* a small model deployment in `NB10_SMALL_MODEL_NAME` (for example `gpt-5.4-mini`) and a
  `gpt-6-luna` deployment in `FOUNDRY_DECISIONS_NAME` for the Decisions API (not available in East US 2).
- *Optional, step 4:* a Model Router deployment in `FOUNDRY_ROUTER_NAME`.
- *Optional, step 11:* notebook 2's hosted agent deployed (`HOSTED_AGENT_NAME`, default `demo-hosted-agent`).
- *Optional, step 12:* notebook 9's Azure AI Search service in `AZURE_SEARCH_ENDPOINT` and its knowledge base
  `games-kb` (or another knowledge base with a model configuration, in `NB10_KNOWLEDGE_BASE`); your account has
  **Search Index Data Reader** on the service and the Fabric access notebook 9 needs for its ontology source.
- Empty optional settings skip their step or row; nothing is created in Azure by this notebook.

| Step | What it shows | Status | Deck slide |
| --- | --- | --- | --- |
| 0 | Settings, clients and the `LIVE` flag | — | — |
| 1 | Time to first token, generation time and output tokens for a short and a long answer | GA | 3, 5 |
| 2 | One question at reasoning effort none, low and medium, with an output limit | GA | 6 |
| 3 | One classification three ways: small model, Decisions API, full model | GA | 8 |
| 4 | Model Router: which model answered, and how fast | GA | 8 |
| 5 | Standard, priority and flex processing side by side, per request | GA; flex preview | 7, 9 |
| 6 | Travel desk scenario, tools and per-turn measurement from framework telemetry | GA | 10, 14 |
| 7 | Direct model call and single agent with three tools | GA | 10, 11 |
| 8 | Sequential and concurrent orchestration | GA | 10, 11 |
| 9 | Hand-off orchestration | GA | 10, 11 |
| 10 | Five patterns compared: time, model calls, tool calls, tokens, answer check | GA | 10, 11, 14 |
| 11 | Hosted agent cold and warm start with the latency debug headers (optional) | GA | 12 |
| 12 | Foundry IQ retrieval effort, minimal and low (optional) | GA; low preview | 13 |
| 13 | Latency budget check: 95th percentile (p95) per pattern against a target | — | 14, 16 |
| 14 | Patterns and features not run here | — | 10, 11, 15 |
| 15 | Cleanup | — | — |

Steps 1–5 are independent of each other. Steps 7–10 need step 6; step 13 needs step 10. Steps 11 and 12 are
independent and optional.

Lines marked `# verify` use a shape that was documented at the time of writing but not executed here — check them against the Learn page linked in the cell before you present.

## 0 · Settings and clients

One `.env` at the repository root, shared with the other kit notebooks. Required: `FOUNDRY_PROJECT_ENDPOINT` and
`FOUNDRY_MODEL_NAME`. Optional settings reuse notebook 1, 2 and 9 names with the same meaning; this notebook adds
`NB10_SMALL_MODEL_NAME` and `NB10_KNOWLEDGE_BASE`. Authentication is Microsoft Entra ID through
`DefaultAzureCredential`; no keys. `LIVE` is `True` only when the required settings are present, and every
chargeable cell checks it.

Source: https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/sdk-overview

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, time, json, asyncio
from urllib.parse import urlparse
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.ai.projects import AIProjectClient
import pandas as pd

load_dotenv(override=True)
env = lambda k, d="": (os.getenv(k) or d).strip()

FOUNDRY_PROJECT_ENDPOINT = env("FOUNDRY_PROJECT_ENDPOINT", "https://<your-foundry-account>.services.ai.azure.com/api/projects/<your-project>")
FOUNDRY_MODEL_NAME       = env("FOUNDRY_MODEL_NAME", "<chat-model-deployment-name>")
# Optional, same meaning as in notebooks 1, 2 and 9; blank skips the step or row
FOUNDRY_DECISIONS_NAME   = env("FOUNDRY_DECISIONS_NAME")   # step 3: gpt-6-luna deployment
FOUNDRY_ROUTER_NAME      = env("FOUNDRY_ROUTER_NAME")      # step 4: Model Router deployment
HOSTED_AGENT_NAME        = env("HOSTED_AGENT_NAME")        # step 11: notebook 2's demo-hosted-agent
AZURE_SEARCH_ENDPOINT    = env("AZURE_SEARCH_ENDPOINT")    # step 12: notebook 9's search service
# Notebook 10 only
NB10_SMALL_MODEL_NAME    = env("NB10_SMALL_MODEL_NAME")    # step 3: small model, e.g. gpt-5.4-mini
NB10_KNOWLEDGE_BASE      = env("NB10_KNOWLEDGE_BASE", "games-kb")   # step 12: notebook 9's knowledge base

_endpoint = urlparse(FOUNDRY_PROJECT_ENDPOINT)
ACCOUNT_NAME = env("FOUNDRY_ACCOUNT_NAME") or (_endpoint.hostname or "").split(".")[0]   # same rule as notebook 1

required = {"FOUNDRY_PROJECT_ENDPOINT": FOUNDRY_PROJECT_ENDPOINT, "FOUNDRY_MODEL_NAME": FOUNDRY_MODEL_NAME}
missing = [k for k, v in required.items() if not v or "<" in v]
print("required settings missing:", ", ".join(missing) if missing else "none")
LIVE = not missing   # chargeable cells run only when the required settings are present
print("optional:", {"step 3 small model": bool(NB10_SMALL_MODEL_NAME), "step 3 Decisions API": bool(FOUNDRY_DECISIONS_NAME),
                    "step 4 router": bool(FOUNDRY_ROUTER_NAME), "step 11 hosted agent": bool(HOSTED_AGENT_NAME),
                    "step 12 knowledge base": bool(AZURE_SEARCH_ENDPOINT)})

credential = DefaultAzureCredential()   # Entra ID by default — no keys
if LIVE:
    project = AIProjectClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential)
    openai_client = project.get_openai_client()   # OpenAI-compatible surface on the project endpoint
    print("Connected to", FOUNDRY_PROJECT_ENDPOINT)

required settings missing: none
optional: {'step 3 small model': True, 'step 3 Decisions API': True, 'step 4 router': True, 'step 11 hosted agent': True, 'step 12 knowledge base': True}
Connected to https://cog-tb7tpjtuee4ji.services.ai.azure.com/api/projects/cog-tb7tpjtuee4ji-project


**Measurement helper for steps 1–5.** `stream_call` streams one Responses API call and records, on this client,
the time to the first visible text token and the time from that token to the end of the stream. The service
reports output tokens (including reasoning tokens), the model that served the request and the service tier in the
final response event. *Milliseconds per visible token* divides the generation time by the visible tokens: an
approximation of the time between tokens, not the Azure Monitor metric.

Sources: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses (streaming) · https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning (usage)

In [3]:
from openai import BadRequestError, RateLimitError

def stream_call(model, prompt, client=None, **kwargs):
    """One streamed Responses API call: client-side timings plus the usage, model and tier the service reports."""
    client = client or openai_client
    t0 = time.perf_counter()
    first, final, parts = None, None, []
    for event in client.responses.create(model=model, input=prompt, stream=True, **kwargs):
        if event.type == "response.output_text.delta":
            first = first or time.perf_counter()
            parts.append(event.delta)
        elif event.type in ("response.completed", "response.incomplete"):   # verify https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses
            final = event.response
    end = time.perf_counter()
    usage = getattr(final, "usage", None)
    out_tokens = getattr(usage, "output_tokens", None)
    reasoning = getattr(getattr(usage, "output_tokens_details", None), "reasoning_tokens", None)
    visible = out_tokens - (reasoning or 0) if out_tokens is not None else None
    generation = end - first if first else None
    return {
        "status": getattr(final, "status", "unknown"),
        "served_by": getattr(final, "model", None),
        "service_tier": getattr(final, "service_tier", None),
        "first_token_s": round(first - t0, 2) if first else None,
        "generation_s": round(generation, 2) if generation is not None else None,
        "total_s": round(end - t0, 2),
        "output_tokens": out_tokens,
        "reasoning_tokens": reasoning,
        "ms_per_visible_token": round(1000 * generation / visible, 1) if generation and visible else None,
        "text": "".join(parts),
    }

TIMING_COLUMNS = ["first_token_s", "generation_s", "total_s", "output_tokens", "reasoning_tokens", "ms_per_visible_token"]

## 1 · Anatomy of one model call  *(slides 3, 5)*

The same deployment answers the same question twice: once in one sentence, once in about 300 words. As per
official documentation, one call takes the time to first token (TTFT: queueing and prompt processing) plus the
time between tokens (TBT) multiplied by the generated tokens, and each generated token adds far more time than
each prompt token. The slide's formula predicts similar first-token times for both answers and a generation time
that grows with the output tokens; the table shows how close one client gets to that. The first call in a kernel
also pays for connection and token set-up, so the table shows medians of three runs.

Sources: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/latency · https://learn.microsoft.com/en-us/azure/foundry/openai/monitor-openai-reference

In [5]:
ANATOMY_PROMPTS = {
    "short": "In one sentence: why does output length drive the latency of a language model call?",
    "long": "In about 300 words, in three short paragraphs: why does output length drive the latency of a language model call?",
}
if LIVE:
    # CHARGEABLE: 6 model calls on FOUNDRY_MODEL_NAME
    rows = []
    for label, prompt in ANATOMY_PROMPTS.items():
        for run in range(1, 4):
            r = stream_call(FOUNDRY_MODEL_NAME, prompt)
            rows.append({"answer": label, "run": run, **{k: r[k] for k in ["status", *TIMING_COLUMNS]}})
    anatomy = pd.DataFrame(rows)
    display(anatomy.groupby("answer", sort=False)[TIMING_COLUMNS].median())
else:
    print("Step 0 settings missing — step 1 skipped")

,first_token_s,generation_s,total_s,output_tokens,reasoning_tokens,ms_per_visible_token
answer,,,,,,
short,1.61,0.60,2.21,29.0,0.0,19.3
long,2.71,7.78,11.11,444.0,56.0,20.6


**What to look for.** The long answer's extra time sits almost entirely in generation time, in proportion to its
extra output tokens. If the model reasons, reasoning tokens arrive before the first visible token and lengthen
TTFT instead. Output caps, low verbosity and concise instructions shorten the generation part; streaming does not
change the total, but lets the reader start at the first token.

## 2 · Reasoning effort and output limits  *(slide 6)*

One short decision at reasoning effort `none`, `low` and `medium`, with `text.verbosity` set to `low` and
`max_output_tokens` set to 600. As per official documentation, supported effort values vary by model; `none`
suits latency-critical work such as classification and voice. The output cap covers reasoning, visible and
formatting tokens: a cap that is too low can return status `incomplete` with no visible text, and the input and
reasoning tokens are still billed. A value the deployment does not accept is recorded as *not accepted*.

Source: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning

In [6]:
EFFORT_QUESTION = ("Travel policy: trips above CHF 2,000 in total need manager pre-approval. A two-day London trip "
                   "costs CHF 830 in flights and CHF 260 for one hotel night. Does it need pre-approval? Answer in one sentence.")
EFFORTS = ["none", "low", "medium"]
MAX_OUTPUT_TOKENS = 600   # covers reasoning and visible tokens
if LIVE:
    # CHARGEABLE: up to 9 model calls on FOUNDRY_MODEL_NAME
    rows = []
    for effort in EFFORTS:
        for run in range(1, 4):
            try:
                r = stream_call(FOUNDRY_MODEL_NAME, EFFORT_QUESTION, reasoning={"effort": effort},
                                text={"verbosity": "low"}, max_output_tokens=MAX_OUTPUT_TOKENS)
            except BadRequestError as e:
                rows.append({"effort": effort, "run": run, "status": f"not accepted ({e.code})", "text": ""})
                break
            rows.append({"effort": effort, "run": run, **{k: r[k] for k in ["status", *TIMING_COLUMNS, "text"]}})
    efforts = pd.DataFrame(rows)
    display(efforts.groupby("effort", sort=False)[[c for c in TIMING_COLUMNS if c in efforts]].median())
    display(efforts.drop_duplicates("effort")[["effort", "status", "text"]])
else:
    print("Step 0 settings missing — step 2 skipped")

,first_token_s,generation_s,total_s,output_tokens,reasoning_tokens,ms_per_visible_token
effort,,,,,,
none,NaN,NaN,NaN,NaN,NaN,NaN
low,1.17,0.50,1.83,31.0,0.0,16.1
medium,2.37,0.63,2.96,90.0,52.0,17.1


,effort,status,text
0,none,not accepted (unsupported_value),
1,low,completed,"No, the trip totals CHF 1,090, which is below ..."
4,medium,completed,"No, the trip totals CHF 1,090, which is below ..."


**What to look for.** Reasoning tokens, first-token time and total time per effort level, and whether every level
gives the same, correct answer (CHF 1,090 needs no pre-approval). The lowest effort that passes evaluation is the
one to ship; for a faster first visible token at higher effort, official documentation suggests prompting for a
short preamble before deeper reasoning.

## 3 · Classification three ways  *(slide 8)*

The travel desk routes each incoming message to one queue: `flight`, `hotel`, `expense_claim`, `visa` or `other`.
Three ways to make the same fixed choice:

- **Small model** (`NB10_SMALL_MODEL_NAME`) at the lowest reasoning effort it accepts, asked for the label only.
- **Decisions API** (`FOUNDRY_DECISIONS_NAME`, `gpt-6-luna`): a `choice` question returns one of the supplied
  values with a confidence, instead of generated text. The call goes to the resource route `/openai/v1/decisions`
  with a Microsoft Entra ID token, as in notebook 1 step 2b.
- **Full model** (`FOUNDRY_MODEL_NAME`) at its default reasoning effort, with the same prompt as the small model.

As per official documentation, the Decisions API is generally available with `gpt-6-luna` and needs OpenAI Python
library 3.26.0 or later (in `requirements.txt`). Thresholds for routing come from labelled examples, not from this
table.

Sources: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/decisions · https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning

In [6]:
from openai import OpenAI

QUEUES = {"flight": "Booking or changing flights.", "hotel": "Hotel bookings and changes.",
          "expense_claim": "Claims and refunds for travel already taken.", "visa": "Visas and entry documents.",
          "other": "Anything else."}
REQUESTS = [
    "Can you move my Thursday flight to London to the evening?",
    "I paid the Paris taxi myself last week, how do I get the money back?",
    "Do I need an ESTA for a two-day trip to Boston?",
    "Please book two nights near our Munich office from 12 March.",
]
LABEL_PROMPT = ("Classify the travel-desk request into exactly one label: " + ", ".join(QUEUES) +
                ". Reply with the label only.\n\nRequest: {request}")

def classify_with_model(model, request, effort=None):
    kwargs = {"reasoning": {"effort": effort}} if effort else {}
    r = stream_call(model, LABEL_PROMPT.format(request=request), max_output_tokens=200, **kwargs)
    return r["text"].strip().lower(), r["total_s"]

def small_model_effort(model):
    """Lowest effort the small model accepts, tried once with a one-word prompt."""
    for effort in ("none", "minimal", "low"):
        try:
            stream_call(model, "Reply with OK.", reasoning={"effort": effort}, max_output_tokens=50)
            return effort
        except BadRequestError:
            continue
    return None

if LIVE:
    decisions_client = None
    if FOUNDRY_DECISIONS_NAME:
        decisions_client = OpenAI(
            base_url=f"https://{ACCOUNT_NAME}.openai.azure.com/openai/v1/",
            api_key=get_bearer_token_provider(credential, "https://ai.azure.com/.default"),   # Entra ID token, refreshed automatically
        )
    # CHARGEABLE: up to 4 requests per method (12) plus up to 3 short calls to find the small model's lowest effort
    small_effort = small_model_effort(NB10_SMALL_MODEL_NAME) if NB10_SMALL_MODEL_NAME else None
    rows = []
    for request in REQUESTS:
        if NB10_SMALL_MODEL_NAME:
            label, seconds = classify_with_model(NB10_SMALL_MODEL_NAME, request, small_effort)
            rows.append({"method": f"small model ({small_effort})", "request": request, "label": label, "seconds": seconds})
        if decisions_client:
            t0 = time.perf_counter()
            decision = decisions_client.decisions.create(
                model=FOUNDRY_DECISIONS_NAME,
                input=request,
                questions=[{"type": "choice", "name": "queue",
                            "instructions": "Which travel-desk queue should handle this request?",
                            "choices": [{"value": v, "description": d} for v, d in QUEUES.items()]}],
            )
            seconds = round(time.perf_counter() - t0, 2)
            answer = decision.answers[0]
            label = f"{answer.choice} ({answer.confidence:.2f})" if answer.type == "choice" else answer.type   # refusal is a valid answer type
            rows.append({"method": "Decisions API", "request": request, "label": label, "seconds": seconds})
        label, seconds = classify_with_model(FOUNDRY_MODEL_NAME, request)
        rows.append({"method": "full model (default effort)", "request": request, "label": label, "seconds": seconds})
    classification = pd.DataFrame(rows)
    display(classification.pivot(index="request", columns="method", values="label"))
    display(classification.groupby("method")["seconds"].agg(["median", "max"]))
else:
    print("Step 0 settings missing — step 3 skipped")

method,Decisions API,full model (default effort),small model (none)
request,,,
Can you move my Thursday flight to London to the evening?,flight (1.00),flight,flight
Do I need an ESTA for a two-day trip to Boston?,visa (1.00),visa,visa
"I paid the Paris taxi myself last week, how do I get the money back?",expense_claim (1.00),expense_claim,expense_claim
Please book two nights near our Munich office from 12 March.,hotel (1.00),hotel,hotel


,median,max
method,,
Decisions API,0.335,1.77
full model (default effort),1.270,1.94
small model (none),1.060,2.23


## 4 · Model Router  *(slide 8)*

Three prompts of rising difficulty go to the Model Router deployment and to the chat deployment. The router is
deployed like any other model and picks an underlying model per prompt (Balanced mode by default); the response
names the model that answered. As per official documentation, routing optimizes cost and latency while keeping
comparable quality, and should be evaluated against the current baseline; the Well-Architected guidance adds
that routers make latency more variable. Skipped when `FOUNDRY_ROUTER_NAME` is empty.

Source: https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/model-router

In [7]:
ROUTER_PROMPTS = {
    "easy": "Which city does the airport code ZRH stand for? One word.",
    "medium": "Summarize in two sentences why travel policies cap hotel prices per city.",
    "hard": ("A team of 6 flies Zurich–London for 2 days. Flights cost CHF 412 out and CHF 418 back per person, hotels "
             "CHF 260 per night, and the budget is CHF 9,000. Which costs can still be added, and how much is left? Show the arithmetic briefly."),
}
if LIVE and FOUNDRY_ROUTER_NAME:
    # CHARGEABLE: 6 model calls (3 on the router, 3 on FOUNDRY_MODEL_NAME)
    rows = []
    for label, prompt in ROUTER_PROMPTS.items():
        for deployment in (FOUNDRY_ROUTER_NAME, FOUNDRY_MODEL_NAME):
            r = stream_call(deployment, prompt)
            rows.append({"prompt": label, "deployment": deployment, **{k: r[k] for k in ["served_by", "first_token_s", "total_s", "output_tokens", "reasoning_tokens"]}})
    display(pd.DataFrame(rows))
elif LIVE:
    print("FOUNDRY_ROUTER_NAME not set — step 4 skipped")
else:
    print("Step 0 settings missing — step 4 skipped")

,prompt,deployment,served_by,first_token_s,total_s,output_tokens,reasoning_tokens
0,easy,model-router,gpt-6-sol-2026-09-22,2.24,2.45,5,0
1,easy,gpt-6.1-sol,gpt-6.1-sol,1.29,1.46,6,0
2,medium,model-router,gpt-6-sol-2026-09-22,2.42,2.71,59,21
3,medium,gpt-6.1-sol,gpt-6.1-sol,1.60,2.65,38,0
4,hard,model-router,gpt-6-sol-2026-09-22,3.07,4.16,312,157
5,hard,gpt-6.1-sol,gpt-6.1-sol,3.15,5.81,274,99


## 5 · Standard, priority and flex processing (flex preview)  *(slides 7, 9)*

The same request three times per service tier on the chat deployment, set per request with `service_tier`:
`default` (standard), `priority` and `flex`. The response reports the tier that actually processed each request.

- **Priority processing** runs on Global Standard and US Data Zone Standard deployments only, needs model version
  2025-12-01 or later, uses the standard quota and bills at the priority rate. Requests can be processed at the
  standard tier after a ramp of more than 50% tokens per minute within 15 minutes, at peak demand, or for long
  context; the response then says `default`.
- **Flex processing (preview)** costs 50% less than standard, has no latency target and no service level agreement,
  and runs on Global Standard. An unsupported model returns HTTP 400; missing flex capacity returns HTTP 429, and
  there is no automatic fallback to standard — the application retries with `default` when completion matters
  more than price.

One idle client is a poor test of priority processing: its latency target is defined in tokens per second across
5-minute windows, so the difference shows under load and in percentiles. The step shows the request-level switch
and the tier the service reports.

Sources: https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/priority-processing · https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/flex-processing

In [8]:
TIER_PROMPT = "List three ways to shorten the response time of a chat assistant, one line each."
if LIVE:
    # CHARGEABLE: up to 9 model calls; priority requests bill at the priority rate, flex at 50% of standard
    rows = []
    for tier in ("default", "priority", "flex"):
        for run in range(1, 4):
            try:
                r = stream_call(FOUNDRY_MODEL_NAME, TIER_PROMPT, service_tier=tier)
                rows.append({"requested": tier, "run": run, "processed_by": r["service_tier"],
                             **{k: r[k] for k in ["first_token_s", "total_s", "output_tokens"]}})
            except BadRequestError as e:   # tier or model not supported for this deployment
                rows.append({"requested": tier, "run": run, "processed_by": f"rejected: HTTP 400 ({e.code})"})
                break
            except RateLimitError:         # flex: capacity unavailable, or quota
                rows.append({"requested": tier, "run": run, "processed_by": "HTTP 429 — no automatic fallback"})
    tiers = pd.DataFrame(rows)
    display(tiers)
    display(tiers.dropna(subset=["total_s"]).groupby("requested", sort=False)[["first_token_s", "total_s"]].agg(["median", "max"]))
else:
    print("Step 0 settings missing — step 5 skipped")

,requested,run,processed_by,first_token_s,total_s,output_tokens
0,default,1,default,3.52,4.51,74
1,default,2,default,1.47,2.52,50
2,default,3,default,1.36,2.68,49
3,priority,1,default,1.68,2.68,56
4,priority,2,default,1.94,2.74,66
5,priority,3,default,1.70,2.42,48
6,flex,1,flex,2.28,3.10,76
7,flex,2,flex,2.12,2.85,58
8,flex,3,flex,1.94,2.77,54


first_token_s       total_s      
                 median   max  median   max
requested                                  
default            1.47  3.52    2.68  4.51
priority           1.70  1.94    2.68  2.74
flex               2.12  2.28    2.85  3.10

## 6 · Travel desk scenario and per-turn measurement  *(slides 10, 14)*

**One question for every pattern.** An employee in Zurich (cost centre CC-4410) asks whether a two-day customer
workshop in London is within policy and budget and which flights to book. Three tools answer the parts:
`get_travel_policy`, `get_remaining_budget` and `find_flights`. They are local functions that return fixed,
made-up data at once and need no approval, so they add no tool latency.

**Same settings for every pattern.** One chat deployment (`FOUNDRY_MODEL_NAME`), one reasoning effort and one
output cap (`ORCH_OPTIONS`), one client. For a model without reasoning effort, remove the `reasoning` entry.

**Measurement.** Microsoft Agent Framework emits OpenTelemetry spans by default: `chat` for each model call and
`execute_tool` for each tool call, with token usage as attributes. An in-memory exporter keeps the spans of one
run, so *model calls per turn* is the number of `chat` spans — the metric slide 14 asks to monitor. Prompts and
answers stay out of the spans. Each pattern runs once as a warm-up (connection and token set-up) and then
`N_RUNS` times; every run has a 180-second deadline.

**Answer check.** A keyword check, not an evaluation: does the answer name an outbound flight, a return flight
and the remaining budget, which only the tools know? Use Foundry evaluators for real quality measurement.

Sources: https://learn.microsoft.com/en-us/agent-framework/agents/observability · https://learn.microsoft.com/en-us/agent-framework/agents/running-agents

In [9]:
from typing import Annotated
from agent_framework import Agent, AgentResponseUpdate, tool

TRAVEL_QUESTION = (
    "I work in Zurich, cost centre CC-4410. I have a customer workshop in London on 3 and 4 March: I fly out on the "
    "morning of 3 March, back on the evening of 4 March, with one hotel night. Is the trip within travel policy and "
    "budget, and which flights should I book? Answer in at most five sentences and name the flight numbers and the remaining budget."
)
POLICY = ("Travel policy TP-7: economy class on flights under six hours; hotels in London up to CHF 260 per night; "
          "trips above CHF 2,000 in total need manager pre-approval.")
BUDGET = "Cost centre CC-4410 has CHF 3,150 left in its Q1 travel budget."
FLIGHTS_OUT = ("3 March, Zurich to London Heathrow: LX 318 departs 07:20, arrives 08:05, economy CHF 412; "
               "LX 324 departs 10:15, arrives 11:00, economy CHF 389.")
FLIGHTS_BACK = ("4 March, London Heathrow to Zurich: LX 339 departs 13:10, arrives 15:55, economy CHF 371; "
                "LX 345 departs 18:40, arrives 21:25, economy CHF 418.")

@tool(approval_mode="never_require")   # read-only lookups on made-up data
def get_travel_policy(destination: Annotated[str, "Destination city"]) -> str:
    """Return the company travel-policy rules that apply to a trip to the destination."""
    return POLICY

@tool(approval_mode="never_require")
def get_remaining_budget(cost_centre: Annotated[str, "Cost centre code, for example CC-4410"]) -> str:
    """Return the remaining travel budget of a cost centre."""
    return BUDGET if cost_centre.strip().upper() == "CC-4410" else f"No travel budget found for {cost_centre}."

@tool(approval_mode="never_require")
def find_flights(origin: Annotated[str, "Origin city"], destination: Annotated[str, "Destination city"],
                 date: Annotated[str, "Travel date, for example 3 March"]) -> str:
    """Return economy flight options for one leg of a trip."""
    return FLIGHTS_OUT if "london" in destination.lower() else FLIGHTS_BACK

EXPECTED = {
    "outbound flight": ("LX 318", "LX318", "LX 324", "LX324"),
    "return flight": ("LX 345", "LX345", "LX 339", "LX339"),
    "remaining budget": ("3,150", "3150", "3'150", "3’150", "3 150"),
}
def answer_check(answer):
    return sum(any(k.lower() in (answer or "").lower() for k in keys) for keys in EXPECTED.values())

In [10]:
from agent_framework.foundry import FoundryChatClient
from agent_framework.observability import enable_instrumentation
from azure.identity.aio import DefaultAzureCredential as AsyncDefaultAzureCredential
from opentelemetry import trace   # verify https://opentelemetry-python.readthedocs.io/en/latest/sdk/trace.export.html
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter   # verify https://opentelemetry-python.readthedocs.io/en/latest/sdk/trace.export.html

ORCH_OPTIONS = {"reasoning": {"effort": "low"}, "max_tokens": 700}   # every pattern; max_tokens  # verify https://learn.microsoft.com/en-us/agent-framework/agents/running-agents
N_WARMUP, N_RUNS, RUN_TIMEOUT_S = 1, 3, 180

if "SPANS" not in globals():   # add the exporter once per kernel
    SPANS = InMemorySpanExporter()
    provider = trace.get_tracer_provider()
    if not isinstance(provider, TracerProvider):   # keep a provider another cell already configured
        provider = TracerProvider()
        trace.set_tracer_provider(provider)
    provider.add_span_processor(SimpleSpanProcessor(SPANS))   # spans reach the exporter as they end
    enable_instrumentation(enable_sensitive_data=False)       # timings and token counts only, no prompts

if LIVE:
    async_credential = AsyncDefaultAzureCredential()
    chat_client = FoundryChatClient(project_endpoint=FOUNDRY_PROJECT_ENDPOINT, model=FOUNDRY_MODEL_NAME,
                                    credential=async_credential)   # one client shared by every agent, as in notebook 9

def spans_of_run():
    spans = SPANS.get_finished_spans()
    chats = [s for s in spans if s.name.startswith("chat")]
    tools = [s for s in spans if s.name.startswith("execute_tool")]
    tokens = sum(int(s.attributes.get("gen_ai.usage.output_tokens") or 0) for s in chats)   # verify https://learn.microsoft.com/en-us/agent-framework/agents/observability
    return len(chats), len(tools), tokens

async def measure(pattern, run_once):
    """Warm-up plus N_RUNS runs of one pattern; returns one row per measured run."""
    rows = []
    for i in range(N_WARMUP + N_RUNS):
        SPANS.clear()
        t0 = time.perf_counter()
        status, out = "completed", {}
        try:
            async with asyncio.timeout(RUN_TIMEOUT_S):
                out = await run_once(t0)
        except TimeoutError:
            status = "timeout"
        except Exception as e:   # recorded in the row so the other patterns still run
            status = f"error: {type(e).__name__}: {str(e)[:120]}"
        total = round(time.perf_counter() - t0, 2)
        calls, tool_calls, tokens = spans_of_run()
        row = {"pattern": pattern, "run": i - N_WARMUP + 1, "status": status,
               "first_text_s": out.get("first_text_s"), "total_s": total,
               "model_calls": out.get("model_calls", calls), "tool_calls": out.get("tool_calls", tool_calls),
               "output_tokens": out.get("output_tokens", tokens),
               "answer_check": f"{answer_check(out.get('answer'))}/3", "route": out.get("route", ""),
               "answer": (out.get("answer") or "")[:400]}
        print(f"{pattern:16s} {'warm-up' if i < N_WARMUP else 'run ' + str(row['run'])}: {status}, "
              f"{total:5.1f} s, {row['model_calls']} model calls, answer {row['answer_check']}")
        if i >= N_WARMUP:
            rows.append(row)
    return rows

def seconds_since(t0, t):
    return round(t - t0, 2) if t else None

RESULTS = {}

## 7 · Direct model call and single agent  *(slides 10, 11)*

**Direct model call.** The application already holds the policy, budget and flight data (fetched in code) and
grounds one Responses API call with it: one model call, no tools. As per official documentation, if prompt
engineering can solve the problem, an agent is not needed.

**Single agent with three tools.** One agent decides which tools to call. Each tool round is another model call:
one call to plan the tool calls (the model can request several at once), one to write the answer, and more if it
plans in several rounds. Official documentation calls a single agent with tools often the right default for
enterprise use cases, with iteration limits against tool-call loops.

Source: https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns

In [11]:
DESK_INSTRUCTIONS = ("You are the employee travel desk. Answer only from the travel systems' data, in at most five "
                     "sentences, and name the flight numbers and the remaining budget.")
GROUNDED_PROMPT = (f"{DESK_INSTRUCTIONS}\n\nTravel systems' data:\n- {POLICY}\n- {BUDGET}\n- {FLIGHTS_OUT}\n- {FLIGHTS_BACK}"
                   f"\n\nEmployee question: {TRAVEL_QUESTION}")

async def run_direct(t0):
    extra = {"reasoning": ORCH_OPTIONS["reasoning"]} if "reasoning" in ORCH_OPTIONS else {}
    r = stream_call(FOUNDRY_MODEL_NAME, GROUNDED_PROMPT, max_output_tokens=ORCH_OPTIONS["max_tokens"], **extra)
    return {"answer": r["text"], "first_text_s": r["first_token_s"], "model_calls": 1, "tool_calls": 0,
            "output_tokens": r["output_tokens"]}

def desk_agent():
    return Agent(client=chat_client, name="travel_desk",
                 instructions=DESK_INSTRUCTIONS + " Use the tools to look up the policy, the budget and the flights.",
                 tools=[get_travel_policy, get_remaining_budget, find_flights], default_options=ORCH_OPTIONS)

async def run_single_agent(t0):
    first, parts = None, []
    async for update in desk_agent().run(TRAVEL_QUESTION, stream=True):
        if update.text:
            first = first or time.perf_counter()
            parts.append(update.text)
    return {"answer": "".join(parts), "first_text_s": seconds_since(t0, first)}

if LIVE:
    # CHARGEABLE: (N_WARMUP + N_RUNS) runs per pattern; about 1 model call per direct run, 2 or more per agent run
    RESULTS["direct model call"] = await measure("direct model call", run_direct)
    RESULTS["single agent"] = await measure("single agent", run_single_agent)
else:
    print("Step 0 settings missing — step 7 skipped")

direct model call warm-up: completed,   8.4 s, 1 model calls, answer 2/3
direct model call run 1: completed,   5.0 s, 1 model calls, answer 3/3
direct model call run 2: completed,   3.4 s, 1 model calls, answer 2/3
direct model call run 3: completed,   5.3 s, 1 model calls, answer 2/3
single agent     warm-up: completed,   7.5 s, 2 model calls, answer 3/3
single agent     run 1: completed,   6.4 s, 2 model calls, answer 3/3
single agent     run 2: completed,   6.8 s, 2 model calls, answer 3/3
single agent     run 3: completed,   7.0 s, 2 model calls, answer 3/3


## 8 · Sequential and concurrent orchestration  *(slides 10, 11)*

The same three specialists, each with one tool, in two Microsoft Agent Framework orchestrations:

- **Sequential** — policy, then budget, then itinerary. By default each agent receives the previous agent's full
  conversation, and only the last agent's output streams as the workflow output, so the user's first answer token
  waits for every stage before it. The itinerary specialist also writes the final answer.
- **Concurrent** — the three specialists run in parallel; a custom aggregator passes their findings to an answer
  writer. The slowest specialist sets the pace, and this aggregator returns its text only when it is complete, so
  the first answer text arrives with the end of the run.

As per official documentation, sequential orchestration has no parallelism and should be avoided for independent
stages; concurrent orchestration reduces run time but is limited by model quota. All five Agent Framework
patterns support streaming.

Sources: https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/sequential · https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/concurrent

In [15]:
from agent_framework import AgentExecutorResponse
from agent_framework.orchestrations import SequentialBuilder, ConcurrentBuilder

SPECIALISTS = {
    "policy_specialist": ("Checks the travel policy.",
                          "You check company travel policy. Call get_travel_policy and state in at most two sentences "
                          "which rules apply to the trip and whether pre-approval is needed.", [get_travel_policy]),
    "budget_specialist": ("Checks the remaining travel budget.",
                          "You check travel budgets. Call get_remaining_budget and state in one sentence the remaining "
                          "budget and whether the trip fits.", [get_remaining_budget]),
    "itinerary_specialist": ("Finds and recommends flights.",
                             "You plan itineraries. Call find_flights for the outbound and the return leg and recommend "
                             "one flight for each leg in at most two sentences.", [find_flights]),
}
FINAL_WRITER = (" You are the last stage: write the final answer to the employee, combining the policy and budget "
                "findings above with your flights, in at most five sentences with the flight numbers and the remaining budget.")

def specialists(extra=None,persist_per_call=False):
    """Fresh agents for each run; extra maps an agent name to text appended to its instructions."""
    extra = extra or {}
    return [Agent(client=chat_client, 
                  name=name, 
                  description=description, 
                  instructions=instructions + extra.get(name, ""),
                  tools=tools, 
                  default_options=ORCH_OPTIONS,
                  require_per_service_call_history_persistence=persist_per_call)
            for name, (description, instructions, tools) in SPECIALISTS.items()]

async def stream_workflow(workflow, t0):
    """Runs a workflow with streaming; returns the answer and the time to its first text."""
    first, by_author, final = None, {}, None
    async for event in workflow.run(TRAVEL_QUESTION, stream=True):
        if event.type != "output":
            continue
        if isinstance(event.data, AgentResponseUpdate) and event.data.text:
            first = first or time.perf_counter()
            author = event.data.author_name or "agent"
            by_author[author] = by_author.get(author, "") + event.data.text
        elif isinstance(event.data, str):   # custom aggregator output
            first = first or time.perf_counter()
            final = event.data
    answer = final if final is not None else (list(by_author.values())[-1] if by_author else "")
    return {"answer": answer, "first_text_s": seconds_since(t0, first)}

async def run_sequential(t0):
    workflow = SequentialBuilder(participants=specialists({"itinerary_specialist": FINAL_WRITER})).build()
    return await stream_workflow(workflow, t0)

async def run_concurrent(t0):
    writer = Agent(client=chat_client, name="answer_writer", default_options=ORCH_OPTIONS,
                   instructions="Combine the specialists' findings into the final answer to the employee, in at most "
                                "five sentences, with the flight numbers and the remaining budget.")

    async def write_answer(results: list[AgentExecutorResponse]) -> str:
        findings = "\n".join(f"{r.executor_id}: {r.agent_response.messages[-1].text if r.agent_response.messages else ''}"
                             for r in results)
        response = await writer.run(f"Employee question: {TRAVEL_QUESTION}\n\nSpecialist findings:\n{findings}")
        return response.text

    workflow = ConcurrentBuilder(participants=specialists()).with_aggregator(write_answer).build()
    return await stream_workflow(workflow, t0)

if LIVE:
    # CHARGEABLE: (N_WARMUP + N_RUNS) runs per pattern; about 6 model calls per sequential run and 7 per concurrent run
    RESULTS["sequential"] = await measure("sequential", run_sequential)
    RESULTS["concurrent"] = await measure("concurrent", run_concurrent)
else:
    print("Step 0 settings missing — step 8 skipped")

sequential       warm-up: completed,  16.1 s, 6 model calls, answer 2/3
sequential       run 1: completed,  15.5 s, 6 model calls, answer 3/3
sequential       run 2: completed,  15.4 s, 6 model calls, answer 2/3
sequential       run 3: completed,  16.0 s, 6 model calls, answer 3/3
concurrent       warm-up: completed,  10.2 s, 7 model calls, answer 3/3
concurrent       run 1: completed,   9.7 s, 7 model calls, answer 3/3
concurrent       run 2: completed,   9.9 s, 7 model calls, answer 3/3
concurrent       run 3: completed,  11.0 s, 7 model calls, answer 3/3


## 9 · Hand-off orchestration  *(slides 10, 11)*

A triage agent receives the question and hands off to the policy specialist, which hands off to the budget
specialist, which hands off to the itinerary specialist; the itinerary specialist answers the employee. Each
hand-off is a tool call that the model makes, so every transfer adds model work, and the agents share the
conversation as it grows. The handoff rules (`add_handoff`) allow only this chain, which keeps the run from
looping between agents.

Hand-off is interactive by default: when an agent replies without handing off, the workflow returns control to the
user with a `request_info` event, which ends the measured turn here. As per official documentation, when the right
specialist is clear from the first message, a deterministic router is better than a chain of hand-offs (step 14).

Source: https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/handoff

In [16]:
from agent_framework.orchestrations import HandoffBuilder, HandoffAgentUserRequest

HANDOFF_NOTES = {
    "policy_specialist": " Then hand off to the budget specialist.",
    "budget_specialist": " Then hand off to the itinerary specialist.",
    "itinerary_specialist": FINAL_WRITER,
}

def handoff_workflow():
    triage = Agent(client=chat_client, name="triage", description="Routes travel requests to the specialists.",
                   instructions="You are the travel desk front office. Do not answer yourself: hand off to the policy specialist.",
                   default_options=ORCH_OPTIONS,
                   require_per_service_call_history_persistence=True)
    policy, budget, itinerary = specialists(HANDOFF_NOTES, persist_per_call=True)
    return (HandoffBuilder(name="travel_desk_handoff", participants=[triage, policy, budget, itinerary])
            .with_start_agent(triage)
            .add_handoff(triage, [policy])
            .add_handoff(policy, [budget])
            .add_handoff(budget, [itinerary])
            .build())

async def run_handoff(t0):
    first, by_author, answer = None, {}, ""
    async for event in handoff_workflow().run(TRAVEL_QUESTION, stream=True):
        if event.type == "output" and isinstance(event.data, AgentResponseUpdate) and event.data.text:
            first = first or time.perf_counter()
            author = event.data.author_name or "agent"
            by_author[author] = by_author.get(author, "") + event.data.text
        elif event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
            messages = event.data.agent_response.messages   # the agent that returned control to the user
            answer = next((m.text for m in reversed(messages) if m.text), "")
    if not answer and by_author:
        answer = list(by_author.values())[-1]
    return {"answer": answer, "first_text_s": seconds_since(t0, first), "route": " → ".join(by_author)}

if LIVE:
    # CHARGEABLE: (N_WARMUP + N_RUNS) runs; about 7 or more model calls per run
    RESULTS["hand-off"] = await measure("hand-off", run_handoff)
else:
    print("Step 0 settings missing — step 9 skipped")

No handoff configuration found for agent 'itinerary_specialist'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


No handoff configuration found for agent 'itinerary_specialist'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


hand-off         warm-up: completed,  19.3 s, 8 model calls, answer 3/3


No handoff configuration found for agent 'itinerary_specialist'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


hand-off         run 1: completed,  18.9 s, 8 model calls, answer 3/3


No handoff configuration found for agent 'itinerary_specialist'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


hand-off         run 2: completed,  19.4 s, 8 model calls, answer 2/3
hand-off         run 3: completed,  17.6 s, 8 model calls, answer 3/3


## 10 · Five patterns compared  *(slides 10, 11, 14)*

Medians of the measured runs per pattern, the slowest run, and the median number of model calls, tool calls and
output tokens per turn. *First text* is the first visible text the user would see: for the single agent and the
hand-off it can be a short remark before the answer; for the concurrent pattern it is the complete answer.

**What to look for.** Total time tracks the model calls made one after another more than the total number of
calls: the concurrent pattern runs its three specialists side by side. Patterns with more calls also produce more
output tokens, and often a wider spread between the median and the slowest run. Check the answer column before comparing
times: a fast pattern that misses a fact is not faster.

In [18]:
if RESULTS:
    runs = pd.DataFrame([row for rows in RESULTS.values() for row in rows])
    done = runs[runs.status == "completed"]
    comparison = done.groupby("pattern", sort=False).agg(
        runs=("run", "count"),
        first_text_median_s=("first_text_s", "median"),
        total_median_s=("total_s", "median"),
        total_slowest_s=("total_s", "max"),
        model_calls=("model_calls", "median"),
        tool_calls=("tool_calls", "median"),
        output_tokens=("output_tokens", "median"),
        answer_check=("answer_check", lambda s: s.mode().iat[0]),
    )
    display(comparison.style.bar(subset=["total_median_s"], color="#9ec5fe").format(precision=1))
    not_completed = runs[runs.status != "completed"]
    if len(not_completed):
        display(not_completed[["pattern", "run", "status"]])
    display(runs.drop_duplicates("pattern")[["pattern", "route", "answer"]])
else:
    print("No results yet — run steps 6 to 9 first")

,runs,first_text_median_s,total_median_s,total_slowest_s,model_calls,tool_calls,output_tokens,answer_check
pattern,,,,,,,,
direct model call,3,2.2,5.0,5.3,1.0,0.0,201.0,2/3
single agent,3,3.9,6.8,7.0,2.0,4.0,274.0,3/3
sequential,3,12.8,15.5,16.0,6.0,4.0,442.0,3/3
concurrent,3,9.9,9.9,11.0,7.0,4.0,597.0,3/3
hand-off,3,4.6,18.9,19.4,8.0,4.0,457.0,3/3


,pattern,route,answer
0,direct model call,,Book economy on LX 324 on 3 March (Zurich–Heat...
3,single agent,,Book economy flights **LX 318** from Zurich on...
6,sequential,,Book economy flight **LX 318** from Zurich to ...
9,concurrent,,"The proposed economy flights are **LX 318**, Z..."
12,hand-off,policy_specialist → budget_specialist → itiner...,Book economy flight **LX 318** from Zurich to ...


## 11 · Hosted agent cold and warm start  *(slide 12)*

Three requests to notebook 2's hosted agent on its dedicated Responses endpoint, each with the header
`x-ms-debug-latency-enabled: true`. The response headers report the session start type (`cold`, `warm` or
`resume`) and a breakdown in milliseconds: platform preprocessing, infrastructure set-up, container readiness and
container response, which add up to `response-begin-ms`. Infrastructure and readiness are omitted for warm
requests. As per official documentation, with the Responses protocol each conversation is associated with a
session, compute follows the session, and idle compute is released after the idle timeout (default 15 minutes).

1. A new conversation — a new session.
2. The same conversation again — the session's compute is still running.
3. A second new conversation — another new session.

The step prints what the service reports; it does not assume a start type. Latency headers are not returned for
WebSocket (`invocations_ws`) requests. Skipped when `HOSTED_AGENT_NAME` is empty; an agent that is not deployed
returns HTTP 404.

Sources: https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/debug-hosted-agent · https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents

In [19]:
import requests

LATENCY_HEADERS = ["session-start-type", "platform-preprocessing-ms", "infra-setup-ms",
                   "container-readiness-ms", "container-response-ms", "response-begin-ms"]

def hosted_post(path, body, debug=True):
    base = f"{FOUNDRY_PROJECT_ENDPOINT}/agents/{HOSTED_AGENT_NAME}/endpoint/protocols/openai"
    headers = {"Authorization": "Bearer " + credential.get_token("https://ai.azure.com/.default").token,
               "Content-Type": "application/json"}
    if debug:
        headers["x-ms-debug-latency-enabled"] = "true"
    t0 = time.perf_counter()
    response = requests.post(f"{base}/{path}", params={"api-version": "v1"}, headers=headers, json=body, timeout=180)
    return response, round(time.perf_counter() - t0, 2)

def new_conversation():
    response, _ = hosted_post("conversations", {}, debug=False)   # verify https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents
    return response.json().get("id") if response.ok else None

if LIVE and HOSTED_AGENT_NAME:
    # CHARGEABLE: 3 hosted-agent turns (session compute plus the agent's model calls) and 2 stored conversations
    rows = []
    first_conversation = new_conversation()
    plan = [("1 · new conversation", first_conversation), ("2 · same conversation", first_conversation),
            ("3 · another new conversation", new_conversation())]
    if not first_conversation:
        print("Conversation not created — each request is sent without a conversation instead.")
    for label, conversation in plan:
        body = {"input": "In one sentence, what are you and where are you running?"}
        if conversation:
            body["conversation"] = conversation
        response, client_s = hosted_post("responses", body)
        if response.status_code == 404:
            print(f"{HOSTED_AGENT_NAME} not found — deploy notebook 2's agents first (agent/README.md)")
            break
        rows.append({"request": label, "http": response.status_code, "client_s": client_s,
                     **{h: response.headers.get(f"x-ms-debug-latency-{h}") for h in LATENCY_HEADERS}})
    if rows:
        display(pd.DataFrame(rows))
elif LIVE:
    print("HOSTED_AGENT_NAME not set — step 11 skipped")
else:
    print("Step 0 settings missing — step 11 skipped")

,request,http,client_s,session-start-type,platform-preprocessing-ms,infra-setup-ms,container-readiness-ms,container-response-ms,response-begin-ms
0,1 · new conversation,200,21.44,cold,196,1514,13427,5840,20977
1,2 · same conversation,200,6.82,warm,88,NaN,NaN,6276,6364
2,3 · another new conversation,200,21.15,cold,128,1335,12815,6246,20524


**What to look for.** On a cold or resumed start, a high `infra-setup-ms` points to platform provisioning and a high
`container-readiness-ms` to the container's own start-up; the fixes are lean start-up and readiness checks and
memory snapshots. On a warm request, `container-response-ms` is the agent's own work: model calls, tool calls and
retries, which the earlier steps measure.

## 12 · Foundry IQ retrieval effort, minimal and low (low preview)  *(slide 13)*

The same question against notebook 9's knowledge base at retrieval reasoning effort `minimal` and `low`, both with
extractive output (`extractiveData`), three times each in alternating order.

- **minimal** disables query planning by a language model: direct text and vector searches across every knowledge
  source in the knowledge base, lowest cost and latency, extractive output only.
- **low** (the default, preview) runs one pass of query planning and knowledge source selection, then fans out
  subqueries and merges the results. It needs a model configuration on the knowledge base.

The table counts references and the activity steps the service reports, and sums the tokens of the planning step.
Notebook 9's `games-kb` includes the Fabric ontology source, which acts on behalf of the signed-in user, so each
request carries your Azure AI Search token as notebook 9 does. The notebook uses the same REST API version as
notebook 9 (`2026-08-01-preview`); as per official documentation, minimal retrieval with extractive output is also
in the generally available REST API.

Source: https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-set-retrieval-reasoning-effort

In [23]:
import re
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient

SEARCH_API_VERSION = "2026-08-01-preview"   # as in notebook 9
KB_QUESTION = "Which procedure applies when wind exceeds the operating limit at an outdoor venue?"   # any notebook 9 question works

def pick(d, name, default=None):   # the SDK can return camelCase or snake_case keys, as handled in notebook 9
    snake = re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()
    return d.get(name, d.get(snake, default)) if isinstance(d, dict) else default

def kb_retrieve(effort):
    body = {
        "retrievalReasoningEffort": {"kind": effort},
        "outputMode": "extractiveData",
        "includeActivity": True,
    }

    if effort == "minimal":
        body["intents"] = [
            {"type": "semantic", "search": KB_QUESTION}
        ]
    else:
        body["messages"] = [{
            "role": "user",
            "content": [{"type": "text", "text": KB_QUESTION}],
        }]   # verify https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-set-retrieval-reasoning-effort
    user_token = credential.get_token("https://search.azure.com/.default").token
    t0 = time.perf_counter()
    result = kb_client.retrieve(retrieval_request=body,
                                headers={"x-ms-query-source-authorization": user_token}).as_dict()
    seconds = round(time.perf_counter() - t0, 2)
    activity = pick(result, "activity", []) or []
    planning = [a for a in activity if "planning" in str(pick(a, "type", "")).lower()]
    return {"effort": effort, "seconds": seconds, "references": len(pick(result, "references", []) or []),
            "activity_steps": len(activity),
            "planning_tokens": sum((pick(a, "inputTokens") or 0) + (pick(a, "outputTokens") or 0) for a in planning)}

if LIVE and AZURE_SEARCH_ENDPOINT:
    kb_client = KnowledgeBaseRetrievalClient(endpoint=AZURE_SEARCH_ENDPOINT, knowledge_base_name=NB10_KNOWLEDGE_BASE,
                                             credential=credential, api_version=SEARCH_API_VERSION)
    # CHARGEABLE: 6 retrieve requests; the low-effort requests also use the knowledge base's model for query planning
    retrieval = pd.DataFrame([kb_retrieve(effort) for _ in range(3) for effort in ("low", "medium")])
    display(retrieval.groupby("effort", sort=False).agg(["median", "max"]))
elif LIVE:
    print("AZURE_SEARCH_ENDPOINT not set — step 12 skipped")
else:
    print("Step 0 settings missing — step 12 skipped")

seconds        references     activity_steps     planning_tokens       
        median    max     median max         median max          median    max
effort                                                                        
low       4.60  23.71       19.0  23            8.0   9          1371.0   1403
medium    5.69   5.77       18.0  18            9.0   9         11652.0  11749

## 13 · Latency budget check  *(slides 14, 16)*

A latency budget turns percentiles into a release gate: targets per user journey and pattern, and a check that
fails when a change pushes a percentile over its target. This cell applies two targets to the step 10 runs — the
95th percentile (p95) of the first visible text and of the total time — and prints pass or fail per pattern.

The targets below are illustrative; set them from the product requirement for the journey. With three runs per
pattern, the p95 is close to the slowest run. A real gate uses many more requests, for example a load test or
production traces in Application Insights with p50, p95 and p99 per agent and model, and reruns evaluations when a
model or prompt changes.

Sources: https://learn.microsoft.com/en-us/azure/foundry/observability/concepts/trace-agent-concept · https://learn.microsoft.com/en-us/azure/well-architected/ai/application-design

In [ ]:
BUDGET_TARGETS = {"first_text_p95_s": 6.0, "total_p95_s": 15.0}   # illustrative targets for the travel-desk journey

if RESULTS:
    done = pd.DataFrame([row for rows in RESULTS.values() for row in rows]).query("status == 'completed'")
    budget = done.groupby("pattern", sort=False).agg(
        runs=("run", "count"),
        first_text_p95_s=("first_text_s", lambda s: s.quantile(0.95)),
        total_p95_s=("total_s", lambda s: s.quantile(0.95)),
    )
    within = (budget.first_text_p95_s <= BUDGET_TARGETS["first_text_p95_s"]) & (budget.total_p95_s <= BUDGET_TARGETS["total_p95_s"])
    budget["verdict"] = within.map({True: "pass", False: "fail"})
    print("targets:", BUDGET_TARGETS)
    display(budget.round(2))
else:
    print("No results yet — run steps 6 to 10 first")

## 14 · Patterns and features not run here  *(slides 10, 11, 15)*

**Group chat and Magentic orchestration.** Both coordinate agents through a manager: group chat in shared rounds,
Magentic by building and revising a plan with progress checks. As per official documentation, group chat does not
fit real-time requirements and works best with three or fewer agents, and Magentic orchestration does not optimize
for speed and is slow to converge, so it should be avoided for time-sensitive work. Both fit background runs
better than a chat turn; this notebook does not time them.

**Deterministic router.** When the specialist is clear from the first message, official documentation recommends
deterministic routing instead of a chain of hand-offs: a classifier such as the step 3 Decisions API picks the
specialist, and code calls only that one agent. The router adds one fast classification call instead of the
triage agent's model calls.

**Where the other latency features are shown**

| Feature | Where in the kit |
| --- | --- |
| Prompt caching | Notebook 4 |
| Semantic caching in the AI gateway | Notebook 8, step 5 |
| Priority processing on one request | Notebook 1, step 2 |
| Decisions API with three question types | Notebook 1, step 2b |
| Tracing to Application Insights | Notebooks 1 and 2, notebook 9 step 6 |
| Provisioned throughput with spillover | Deck slides 7, 9 and 15 only — a capacity purchase, not a notebook step |
| Asynchronous content filter | Deck slide 6 only — set on the deployment's guardrail; the client must remove flagged text |

Source: https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns

## 15 · Cleanup

The cell closes the Agent Framework chat client and the asynchronous credential and clears the in-memory spans.
Nothing in Azure is deleted, because the notebook creates nothing there except stored records:

- **Not deleted, still billable while they exist:** the model deployments (chat, small model, `gpt-6-luna`, Model
  Router), notebook 2's hosted agent (session compute is released after the idle timeout, 15 minutes by default),
  and notebook 9's Azure AI Search service, which bills per hour on Basic and higher tiers.
- **Not deleted, stored records:** the two conversations created in step 11 (hosted-agent sessions are deleted
  after 30 days of inactivity) and the Responses API responses of steps 1–9, which are retained for 30 days by
  default.

In [ ]:
if "chat_client" in globals():
    await chat_client.close()
if "async_credential" in globals():
    await async_credential.close()   # verify https://learn.microsoft.com/en-us/python/api/azure-identity/azure.identity.aio.defaultazurecredential
if "SPANS" in globals():
    SPANS.clear()
print("Clients closed; nothing was deleted in Azure.")

## What you built

| Step | Built / shown | Azure resource touched |
| --- | --- | --- |
| 1 | Time to first token, generation time and tokens for a short and a long answer | Chat model deployment |
| 2 | Reasoning effort none, low and medium with an output cap and low verbosity | Chat model deployment |
| 3 | One classification by a small model, the Decisions API and the full model | Small model, `gpt-6-luna` and chat deployments |
| 4 | Served model and timings through Model Router | Model Router and chat deployments |
| 5 | Requested and processed service tier per request | Chat model deployment |
| 6 | Travel desk tools, in-memory spans and the measurement harness | None (local) |
| 7 | Direct model call and single agent with three tools | Chat model deployment |
| 8 | Sequential and concurrent orchestrations | Chat model deployment |
| 9 | Hand-off orchestration with a fixed hand-off chain | Chat model deployment |
| 10 | Comparison of the five patterns | None (local) |
| 11 | Session start type and latency breakdown of a hosted agent | Notebook 2's hosted agent |
| 12 | Minimal and low retrieval effort on one knowledge base | Notebook 9's Azure AI Search knowledge base |
| 13 | p95 per pattern against a latency budget | None (local) |

## Sources and tested versions

Written on 14 November 2026 against these versions; **not executed against Azure**.

| Package | Version written against | Used in |
| --- | --- | --- |
| `openai` | 3.26.0 or later (kit floor) | Steps 1–5, 7 |
| `azure-ai-projects` | 2.10.0 (kit range `>=2.9.0,<2.11`) | Step 0 |
| `agent-framework` | 1.25.0 (PyPI, 7 November 2026), through the kit's agent requirements | Steps 6–10 |
| `agent-framework-foundry` | 1.16.2 (PyPI, 12 November 2026), through the kit's agent requirements | Steps 6–10 |
| `opentelemetry-sdk` | 1.27.0 or later (kit floor) | Step 6 |
| `azure-search-documents` | 12.1.0b2 (kit pin) | Step 12 |
| `requests` | 2.32.0 or later (kit floor) | Step 11 |

Microsoft Learn pages used:
- https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/latency
- https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses
- https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning
- https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/decisions
- https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/model-router
- https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/priority-processing
- https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/flex-processing
- https://learn.microsoft.com/en-us/azure/foundry/openai/monitor-openai-reference
- https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns
- https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/sequential
- https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/concurrent
- https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/handoff
- https://learn.microsoft.com/en-us/agent-framework/agents/middleware
- https://learn.microsoft.com/en-us/agent-framework/agents/observability
- https://learn.microsoft.com/en-us/agent-framework/agents/running-agents
- https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/debug-hosted-agent
- https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-set-retrieval-reasoning-effort

Other sources: https://pypi.org/project/agent-framework/ · https://pypi.org/project/agent-framework-foundry/ ·
https://opentelemetry-python.readthedocs.io/en/latest/sdk/trace.export.html

## Before you build on this: re-check …

- **Flex processing (preview):** the supported model list and the HTTP 400 and 429 behaviour change as models are
  added; check the flex page before step 5.
- **Reasoning effort per model:** which deployments accept `none` (step 2) and which effort the small model accepts
  (step 3).
- **`# verify` lines:** the final streaming event types, the `max_tokens` option on `FoundryChatClient`, the token
  attribute on `chat` spans, the in-memory exporter import, the hosted-agent conversation route, the minimal
  retrieval request body and the asynchronous credential's `close`.
- **Agent Framework versions:** `agent-framework-foundry` declares support for `azure-ai-projects>=2.2.0,<2.8.0`,
  while the kit requires `>=2.9.0,<2.11`; check which version the kit's environment resolves (`pip show
  agent-framework-foundry`, `pip check`) before relying on steps 6–10.
- **Foundry IQ:** the GA REST API version for minimal retrieval, and whether `low` stays in preview.
- **Hosted agents:** the latency header names and the idle-timeout default.